# AI-Enhanced Interview Management System
## Machine Learning Module — Random Forest Classifier
### Candidate Selection Prediction

**Algorithm:** Random Forest Classifier  
**Task:** Predict candidate selection probability  
**Output:** Highly Likely / Moderately Likely / Low Selection Probability

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
import joblib
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully')

## Step 2 — Load Dataset

In [ ]:
df = pd.read_csv('recruitment_dataset.csv')

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
print('Dataset Info:')
print(df.info())
print('\nMissing Values:')
print(df.isnull().sum())
print('\nBasic Statistics:')
df.describe()

In [ ]:
# Selection outcome distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
counts = df['selection_outcome'].value_counts()
axes[0].pie(counts, labels=['Not Selected', 'Selected'], autopct='%1.1f%%',
            colors=['#fee2e2', '#d1fae5'], startangle=90)
axes[0].set_title('Candidate Selection Distribution', fontsize=13, fontweight='bold')

# Bar chart - experience vs selection
df.groupby(['years_experience', 'selection_outcome']).size().unstack().plot(
    kind='bar', ax=axes[1], color=['#fee2e2', '#d1fae5'], edgecolor='gray'
)
axes[1].set_title('Experience vs Selection Outcome', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Years of Experience')
axes[1].set_ylabel('Count')
axes[1].legend(['Not Selected', 'Selected'])

plt.tight_layout()
plt.savefig('eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA charts saved')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 7))
corr = df.drop('candidate_id', axis=1).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Correlation heatmap saved')

## Step 4 — Data Preprocessing

In [ ]:
# Drop candidate_id (not a feature)
df_model = df.drop('candidate_id', axis=1)

# Features and target
X = df_model.drop('selection_outcome', axis=1)
y = df_model['selection_outcome']

print('Features:', list(X.columns))
print('Target distribution:')
print(y.value_counts())
print('\nFeature matrix shape:', X.shape)

In [ ]:
# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set size: {X_train.shape[0]}')
print(f'Testing set size:  {X_test.shape[0]}')

## Step 5 — Train Random Forest Model

In [ ]:
# Initialize Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_split=3,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

# Train the model
rf_model.fit(X_train, y_train)
print('✅ Random Forest model trained successfully')
print(f'Number of trees: {rf_model.n_estimators}')
print(f'Number of features: {rf_model.n_features_in_}')

## Step 6 — Model Evaluation

In [ ]:
# Predictions
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

# Metrics
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print('=' * 45)
print('       RANDOM FOREST MODEL PERFORMANCE')
print('=' * 45)
print(f'  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print('=' * 45)
print('\nDetailed Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Selected', 'Selected']))

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Selected', 'Selected'],
            yticklabels=['Not Selected', 'Selected'], ax=axes[0])
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Feature importance
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=True)
feat_imp.plot(kind='barh', color='#4f46e5', ax=axes[1])
axes[1].set_title('Feature Importance', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Evaluation charts saved')

## Step 7 — Prediction Function (3-class output)

In [ ]:
def predict_candidate(years_exp, technical, communication, problem_solving,
                       num_interviews, education, prev_companies, skills_count):
    """
    Predict candidate selection probability.
    Returns: label, probability%, and category
    """
    features = np.array([[years_exp, technical, communication, problem_solving,
                           num_interviews, education, prev_companies, skills_count]])
    
    prob = rf_model.predict_proba(features)[0][1]  # probability of selection
    prob_pct = round(prob * 100, 1)
    
    if prob >= 0.70:
        label = 'Highly Likely to Select'
        color = '🟢'
    elif prob >= 0.40:
        label = 'Moderately Likely to Select'
        color = '🟡'
    else:
        label = 'Low Selection Probability'
        color = '🔴'
    
    return label, prob_pct, color

# Test predictions
print('SAMPLE PREDICTIONS')
print('=' * 50)

test_cases = [
    (5, 9, 8, 9, 3, 4, 3, 8, 'Senior Developer'),
    (1, 5, 5, 5, 1, 2, 1, 3, 'Junior Developer'),
    (3, 7, 7, 7, 2, 3, 2, 6, 'Mid-level Developer'),
]

for case in test_cases:
    *features, name = case
    label, prob, color = predict_candidate(*features)
    print(f'{color} {name}')
    print(f'   Probability: {prob}% — {label}')
    print()

## Step 8 — Save Model

In [ ]:
# Save the trained model
joblib.dump(rf_model, 'random_forest_model.pkl')
print('✅ Model saved as random_forest_model.pkl')

# Save feature names for backend use
feature_names = list(X.columns)
joblib.dump(feature_names, 'feature_names.pkl')
print('✅ Feature names saved')

# Save accuracy report
report = f"""RANDOM FOREST MODEL — ACCURACY REPORT
==========================================
Model       : Random Forest Classifier
Task        : Candidate Selection Prediction
Dataset     : recruitment_dataset.csv
Samples     : {len(df)} total ({len(X_train)} train, {len(X_test)} test)
Features    : {list(X.columns)}

PERFORMANCE METRICS
-------------------
Accuracy    : {accuracy*100:.2f}%
Precision   : {precision*100:.2f}%
Recall      : {recall*100:.2f}%
F1-Score    : {f1*100:.2f}%

OUTPUT CLASSES
--------------
Highly Likely      : probability >= 70%
Moderately Likely  : probability 40-69%
Low Probability    : probability < 40%
"""

with open('model_accuracy_report.txt', 'w') as f:
    f.write(report)

print('✅ Accuracy report saved as model_accuracy_report.txt')
print()
print(report)